# Batch export element maps

%pip install -U git+https://github.com/fligt/maxrf4u.git

In [1]:
import maxrf4u as mx
from glob import glob
import os
import matplotlib.pyplot as plt
import re

In [2]:
mx.__version__

'0.1.42'

In [3]:
os.chdir('../../../data/interim')

In [4]:
datastack_files = glob('**/*.datastack', recursive=True)
for i, f in enumerate(datastack_files): 
    print(f'[{i}] {f}')

[0] maxrf/datastacks/WM-71803-01_400_600_50.datastack
[1] maxrf/datastacks/WM-71803-03_250_300_50.datastack
[2] maxrf/datastacks/WM-71803-08_250_300_50.datastack
[3] maxrf/datastacks/WM-71803-10_250_300_50.datastack
[4] maxrf/datastacks/WM-71803-12_250_300_50.datastack
[5] maxrf/datastacks/WM-71803-13_400_600_50.datastack
[6] maxrf/datastacks/WM-71803-17_400_600_50.datastack
[7] maxrf/datastacks/WM-71803-18_400_300_50_det.datastack
[8] maxrf/datastacks/WM-71803-18_400_500_50.datastack
[9] maxrf/datastacks/WM-71803-19_400_600_50.datastack
[10] maxrf/datastacks/WM-71803-23_400_600_50.datastack
[11] maxrf/datastacks/WM-71803-24_400_600_50.datastack
[12] maxrf/datastacks/WM-71803-29_400_500_50.datastack
[13] maxrf/datastacks/WM-71803-30_250_300_50.datastack
[14] maxrf/datastacks/WM-71803-31_400_600_50.datastack
[15] maxrf/datastacks/WM-71803-35_400_500_50.datastack


In [19]:
folders = [d.removesuffix('.datastack').removeprefix('maxrf/datastacks/') for d in datastack_files]
folders

['WM-71803-01_400_600_50',
 'WM-71803-03_250_300_50',
 'WM-71803-08_250_300_50',
 'WM-71803-10_250_300_50',
 'WM-71803-12_250_300_50',
 'WM-71803-13_400_600_50',
 'WM-71803-17_400_600_50',
 'WM-71803-18_400_300_50_det',
 'WM-71803-18_400_500_50',
 'WM-71803-19_400_600_50',
 'WM-71803-23_400_600_50',
 'WM-71803-24_400_600_50',
 'WM-71803-29_400_500_50',
 'WM-71803-30_250_300_50',
 'WM-71803-31_400_600_50',
 'WM-71803-35_400_500_50']

In [26]:
object_nums

['WM-71803-01',
 'WM-71803-03',
 'WM-71803-08',
 'WM-71803-10',
 'WM-71803-12',
 'WM-71803-13',
 'WM-71803-17',
 'WM-71803-18',
 'WM-71803-18',
 'WM-71803-19',
 'WM-71803-23',
 'WM-71803-24',
 'WM-71803-29',
 'WM-71803-30',
 'WM-71803-31',
 'WM-71803-35']

In [22]:
out_dir = 'maxrf/element-maps/' 

for i, datastack_file in enumerate(datastack_files):
    print(f'{i}/15', end='\r')
    export_element_maps(datastack_file, output_dir=f'{out_dir}{folders[i]}/', histeq=False, verbose=True)

In [21]:
def export_element_maps(datastack_file, output_dir=None, histeq=True, verbose=False):
    '''Quick fix of double colon bug'''

    ds = mx.DataStack(datastack_file)
 
    if not output_dir:
        output_dir = ""
 
    if not os.path.isdir(output_dir):
        os.makedirs(output_dir)

    file_name = os.path.basename(datastack_file).removesuffix(mx.DATASTACK_EXT)
    element_maps = ds.read('nmf_elementmaps')
    element_nums = ds.read('nmf_atomnums')
    elements = mx.elems_from_atomnums(element_nums)
 
    if histeq:   
        element_maps = [ske.equalize_hist(m) for m in element_maps]
 
    for im, elem in zip(element_maps, elements):
        plt.imsave(f'{output_dir}{file_name}_{elem}.png', im)

        if verbose: 
            print(f'{output_dir}{file_name}_{elem}.png saved           ', end='\r')

In [24]:
ds.tree()

/
├── compton_peak_energy (1,) float64
├── hotmax_baselines (32, 4096) float64
├── hotmax_noiselines (32, 4096) float64
├── hotmax_peak_idxs_flat (33,) int64
├── hotmax_peak_idxs_list (32, 2) int64
├── hotmax_spectra (32, 4096) float32
├── hotmax_spots (32, 2) int64
├── hotmax_subpeak_idxs_list (32, 19) int64
├── imvis_extent (4,) int64
├── imvis_reg (583, 355, 3) uint8
├── imvis_reg_ (583, 355, 3) uint8
├── imvis_reg_highres (8256, 5027, 3) uint8
├── imvis_reg_highres_ (8256, 5027, 3) uint8
├── maxrf_cube (583, 355, 4096) float32
├── maxrf_energies (4096,) float64
├── maxrf_maxspectrum (4096,) float32
├── maxrf_sumspectrum (4096,) float64
├── nmf_atomnums (18,) int64
├── nmf_elementmaps (18, 583, 355) float32
├── nmf_gausscomponents (34, 4096) float32
├── nmf_peakmaps (34, 583, 355) float32
└── nmf_peaks2elements_matrix (18, 33) float32

maxrf/datastacks/WM-71803-01_400_600_50.datastack:




In [29]:
for i, [datastack_file, num] in enumerate(zip(datastack_files, object_nums)):

    ds = mx.DataStack(datastack_file)
    imvis_reg = ds.read('imvis_reg')
    imvis_reg_highres = ds.read('imvis_reg_highres')

    output_dir = 'maxrf/element-maps/' + folders[i] + '/'

    print(f'{i}/{len(datastack_files)} Saving RGB images to: {output_dir}...')
    plt.imsave(output_dir + num + '_imvis_reg_highres.png', imvis_reg_highres)
    plt.imsave(output_dir + num + '_imvis_reg.png', imvis_reg)

0/16 Saving RGB images to: maxrf/element-maps/WM-71803-01_400_600_50/...
1/16 Saving RGB images to: maxrf/element-maps/WM-71803-03_250_300_50/...
2/16 Saving RGB images to: maxrf/element-maps/WM-71803-08_250_300_50/...
3/16 Saving RGB images to: maxrf/element-maps/WM-71803-10_250_300_50/...
4/16 Saving RGB images to: maxrf/element-maps/WM-71803-12_250_300_50/...
5/16 Saving RGB images to: maxrf/element-maps/WM-71803-13_400_600_50/...
6/16 Saving RGB images to: maxrf/element-maps/WM-71803-17_400_600_50/...
7/16 Saving RGB images to: maxrf/element-maps/WM-71803-18_400_300_50_det/...
8/16 Saving RGB images to: maxrf/element-maps/WM-71803-18_400_500_50/...
9/16 Saving RGB images to: maxrf/element-maps/WM-71803-19_400_600_50/...
10/16 Saving RGB images to: maxrf/element-maps/WM-71803-23_400_600_50/...
11/16 Saving RGB images to: maxrf/element-maps/WM-71803-24_400_600_50/...
12/16 Saving RGB images to: maxrf/element-maps/WM-71803-29_400_500_50/...
13/16 Saving RGB images to: maxrf/element-ma

In [ ]:
plt.imsave(f'{out_dir}')